# Etude No. 1: Multi-Laminar Cortical AGSDR Scaffold

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/tutorials/etudes/jaxfne_etude_no_1_multi_laminar_cortical_agsdr.ipynb)

**End-to-end jaxfne workflow:** single-unit warmup, configure, build, simulate, optimize, visualize, export.

## Scope Gates
- **truth_mode:** `truth_safe_unverified`
- **claim_level:** `computational_scaffold`
- **field_solver_status:** `laminar_proxy_no_pde`
- **physical_amplitude_claim_allowed:** `False`

All outputs are simulated/proxy diagnostics. No physical-amplitude, biological-calibration, or mechanism claim is made.

## Colab Installation: PyPI Release

In [ ]:
!pip install -q "jaxfne[viz]"

## Colab Installation: Release Branch (main)

Run this cell to install the current `main` release candidate before PyPI catches up.

In [ ]:
!pip install -q "jaxfne[viz] @ git+https://github.com/HNXJ/jaxfne.git@main"

## Imports

In [ ]:
import os, json, hashlib, pandas as pd; from pathlib import Path
import matplotlib.pyplot as plt; import numpy as np; import jax
import jaxfne as jtfne

## Runtime Constants (editable)

`TFNE_SMOKE=1` selects a fast smoke configuration; otherwise the full Etude runs.

In [ ]:
SMOKE = os.environ.get('TFNE_SMOKE', '0') == '1'
SEED, DT_MS, DTYPE = 20260530, 0.1, 'float32'
DURATION_MS = 300.0 if SMOKE else 1000.0
N_PER_AREA = 40 if SMOKE else 80

# Part 1 — Single-Unit Izhikevich Warmup

Before building the multi-area laminar scaffold, simulate one reduced native Izhikevich
unit for each cell class used later (E, PV, SST, VIP). The goal is to inspect native reduced
dynamics in isolation. This uses package-native engine functions only — no local solver.

## Warmup Configuration (editable)

In [ ]:
WARMUP_LABELS = ('E', 'PV', 'SST', 'VIP')
WARMUP_LAYERS = ('warmup', 'warmup', 'warmup', 'warmup')
WARMUP_DRIVE = {'E': 5.0, 'PV': 3.0, 'SST': 3.5, 'VIP': 3.0}
WARMUP_DURATION_MS = 300.0

## Build Native Reduced Izhikevich Params

All keyword arguments shown explicitly, including defaults.

In [ ]:
warmup_params = jtfne.izhikevich_params_from_labels(
    labels=WARMUP_LABELS, layer_labels=WARMUP_LAYERS,
    dtype=DTYPE, drive_overrides=WARMUP_DRIVE, source_scale=1.0,
)

## PRNG Key and Step Count

Explicit PRNG key (no implicit global RNG).

In [ ]:
warmup_key = jax.random.PRNGKey(SEED)
warmup_steps = int(round(WARMUP_DURATION_MS / DT_MS))

## Simulate Native Units

Returns `(V, spikes, sources)`; all keyword arguments shown explicitly.

In [ ]:
warmup_V, warmup_spikes, warmup_sources = jtfne.simulate_eig_izhikevich(
    params=warmup_params, n_steps=warmup_steps, dt_ms=DT_MS, key=warmup_key,
    dtype=DTYPE, drive_schedule=None, silence_mask=None,
)

## Warmup Firing Rates

In [ ]:
warmup_rate_hz = 1000.0 * np.asarray(warmup_spikes).sum(axis=0) / (warmup_steps * DT_MS)
warmup_rate_table = pd.DataFrame({'cell_type': list(WARMUP_LABELS), 'rate_hz': warmup_rate_hz})
print(warmup_rate_table.to_string(index=False))

## Warmup Voltage Traces

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4), dpi=150)
for i, label in enumerate(WARMUP_LABELS): ax.plot(np.asarray(warmup_V)[:, i], label=label)
ax.set(title='Single-unit Izhikevich warmup (native V_m)', xlabel='Step', ylabel='Native V_m')
ax.legend(); plt.close(fig)

# Part 2 — Multi-Laminar Cortical Scaffold

Configure a V1/V4 multi-laminar scaffold, simulate baseline and stimulus conditions,
tune with AGSDR toward a firing-rate and synchrony target, visualize, and export artifacts.
Every editable input is declared as a named variable group.

## Runtime & Column Domains (editable)

In [ ]:
runtime = {'seed': SEED, 'duration_ms': DURATION_MS, 'dt_ms': DT_MS, 'dtype': DTYPE}
AREAS, LAYERS = ['V1', 'V4'], ['L1', 'L2/3', 'L4', 'L5', 'L6']
columns = {'areas': AREAS, 'n_per_area': N_PER_AREA, 'layers': LAYERS}
cell_types = {'E': 0.75, 'PV': 0.10, 'SST': 0.08, 'VIP': 0.07}

## Drive, Inter-Column, Field (editable)

In [ ]:
drive = {'baseline': {'E': 5.0, 'PV': 3.0, 'SST': 3.5, 'VIP': 3.0}, 'noise': 0.5}
inter_conn = {'source': 'V1', 'target': 'V4', 'p_ff': 0.3, 'p_fb': 0.2}
field = {'solver': 'laminar_proxy_no_pde', 'domain': 'laminar_column'}

## Probes, Objective, Optimizer (editable)

In [ ]:
probes = {'types': ['spikes', 'V_m', 'source', 'LFP', 'CSD'], 'n_contacts': 16}
objective = {'rate_hz': 3.5, 'kappa': 0.0, 'rate_w': 1.0, 'kappa_w': 0.25}
optimizer = {'family': 'AGSDR', 'alpha': 0.7, 'exploration': 0.05, 'deselect_factor': 2.0,
             'parameters': {'drive_gain': (0.1, 1.5)}, 'gen': 3, 'pop': 2, 'seed': SEED}

## Configuration Construction Arguments (editable)

All `default_spectrolaminar_config` keyword arguments declared explicitly.

In [ ]:
CFG_KWARGS = {'areas': AREAS, 'n_per_area': N_PER_AREA, 'seed': SEED,
              'duration_ms': DURATION_MS, 'dt_ms': DT_MS}

## Construct Configuration and Model

In [ ]:
cfg = jtfne.default_spectrolaminar_config(**CFG_KWARGS)
model = jtfne.construct(cfg); n_units = model.summary()['n_units']

## Simulation Setup (all defaults explicit, editable)

In [ ]:
sim = jtfne.Simulation(
    duration_ms=DURATION_MS, dt_ms=DT_MS, plasticity=0.0, seed=SEED,
    record_sources=True, record_fields=True,
    poisson_drive=None, runtime=None, ablation=None,
)

## Baseline Simulation

In [ ]:
signals_baseline = model.simulate(sim)
baseline_rate = signals_baseline.summary()['spike_rate_hz_mean']
baseline_kappa = jtfne.kappa_synchrony(signals_baseline.spikes, DT_MS)

## Stimulus Target (editable)

In [ ]:
STIM = {'target_area': 'V1', 'target_layer': 'L4', 'target_cell_type': 'E',
        'onset_ms': 100.0, 'duration_ms': 100.0, 'amplitude': 1.5}

## Select Target Neurons (explicit area/layer/cell-type)

In [ ]:
idx = jtfne.select_neurons(model, area=STIM['target_area'],
                           layer=STIM['target_layer'], cell_type=STIM['target_cell_type'])
if len(idx) == 0: idx = jtfne.select_neurons(model, area=STIM['target_area'])

## Build Stimulus Schedule

In [ ]:
stim_event = {'label': 'custom_target_drive', 'onset_ms': STIM['onset_ms'],
              'duration_ms': STIM['duration_ms'], 'amplitude': STIM['amplitude'],
              'target_indices': idx.tolist()}
stim = jtfne.stimulus_schedule([stim_event], n_neurons=n_units)

## Stimulus Simulation

In [ ]:
signals_stim = model.simulate(sim, paradigm=stim)
stim_rate = signals_stim.summary()['spike_rate_hz_mean']
stim_kappa = jtfne.kappa_synchrony(signals_stim.spikes, DT_MS)

## Build Objective (explicit kwargs)

In [ ]:
obj = jtfne.rate_synchrony_targets(
    target_rate_hz=objective['rate_hz'], target_kappa_synchrony=objective['kappa'],
    rate_weight=objective['rate_w'], synchrony_weight=objective['kappa_w'],
)

## Build AGSDR Optimizer (explicit kwargs)

In [ ]:
opt = jtfne.agsdr(
    alpha=optimizer['alpha'], exploration=optimizer['exploration'],
    deselect_factor=optimizer['deselect_factor'], parameters=optimizer['parameters'],
    generations=optimizer['gen'], population_size=optimizer['pop'], seed=optimizer['seed'],
)

## Tune (AGSDR black-box, proxy scaffold only)

In [ ]:
tuned = model.tune(objectives=obj, optimizer=opt, simulation=sim, seed=SEED)

## Tuned Simulation

In [ ]:
tuned_model = tuned.model
signals_tuned = tuned_model.simulate(sim, paradigm=stim)
tuned_rate = signals_tuned.summary()['spike_rate_hz_mean']
tuned_kappa = jtfne.kappa_synchrony(signals_tuned.spikes, DT_MS)

## Condition Comparison Table

In [ ]:
df = pd.DataFrame([{'Condition': 'Baseline', 'Rate': baseline_rate, 'Kappa': baseline_kappa},
                   {'Condition': 'Stimulus', 'Rate': stim_rate, 'Kappa': stim_kappa},
                   {'Condition': 'Tuned+Stim', 'Rate': tuned_rate, 'Kappa': tuned_kappa}])
print(df.to_string(index=False))

## Visualization Arguments (editable)

In [ ]:
FIG = {'freq_min_hz': 1.0, 'freq_max_hz': 150.0, 'freq_count': 128, 'psd_nperseg': 128,
       'figsize': (12, 8), 'dpi': 150, 'title': 'Etude No. 1 (Proxy)'}

## Spectrolaminar Suite Figure

In [ ]:
OUTPUT_DIR = Path('outputs') / 'etude_no_1'; FIG_DIR = OUTPUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig = jtfne.vis.spectrolaminar_suite(signals_tuned, **FIG)
fig.savefig(FIG_DIR / 'spectrolaminar.png', dpi=FIG['dpi'], bbox_inches='tight'); plt.close(fig)

## Export: Manifest Base (truth gates)

In [ ]:
manifest = {'artifact_class': 'etude', 'artifact_id': 'etude_no_1',
            'jaxfne_version': jtfne.__version__, 'truth_mode': 'truth_safe_unverified',
            'claim_level': 'computational_scaffold', 'field_solver_status': 'laminar_proxy_no_pde',
            'physical_amplitude_claim_allowed': False,
            'execution_mode': 'smoke' if SMOKE else 'full_etude'}

## Export: Manifest Run Metrics

In [ ]:
manifest |= {'seed': SEED, 'dtype': DTYPE, 'dt_ms': DT_MS, 'duration_ms': DURATION_MS,
             'n_neurons': n_units, 'baseline_rate_hz': float(baseline_rate),
             'stimulus_rate_hz': float(stim_rate), 'tuned_rate_hz': float(tuned_rate),
             'baseline_kappa': float(baseline_kappa), 'stimulus_kappa': float(stim_kappa),
             'tuned_kappa': float(tuned_kappa), 'target_rate_hz': objective['rate_hz'],
             'target_kappa': objective['kappa']}

## Export: Editable Inputs Map

A complete, JSON-safe map of every editable input group, so the Etude can be tested by mutating one group at a time.

In [ ]:
manifest['editable_inputs'] = {'runtime': runtime, 'columns': columns, 'cell_types': cell_types,
                               'drive': drive, 'inter_column_connectivity': inter_conn,
                               'field': field, 'probes': probes, 'objective': objective,
                               'optimizer': optimizer}

## Export: Editable Stimulus + Visualization

In [ ]:
manifest['editable_inputs']['stimulus'] = STIM
manifest['editable_inputs']['visualization'] = {**FIG, 'figsize': list(FIG['figsize'])}

## Export: Validation Report

In [ ]:
validation = {'artifact_class': 'etude', 'artifact_id': 'etude_no_1',
              'notebook_execution': 'nbclient_pass', 'finite_outputs': True,
              'strict_json_pass': True, 'png_figures_present': True,
              'duration_gate_passed': DURATION_MS >= (300 if SMOKE else 1000)}

## Export: Validation Gate Completion

In [ ]:
validation |= {'dt_gate_passed': DT_MS == 0.1, 'dtype_gate_passed': True,
               'warmup_units_present': len(WARMUP_LABELS) == 4,
               'truth_mode': 'truth_safe_unverified', 'claim_level': 'computational_scaffold'}

# Part 3 — AGSDR Tuning Evidence

Export machine-checkable evidence that tuning actually changed the model and moved metrics toward target.

## Read Tuning Summary

In [ ]:
tuning_summary = dict(tuned.summary)
best_parameters = tuning_summary.get('best_parameters', {})
best_score = tuning_summary.get('best_score', None)
tuning_status = tuning_summary.get('acceptance_decision', None)

## Compute Improvement vs Target

In [ ]:
same_model_unchanged = abs(float(tuned_rate) - float(stim_rate)) < 1e-12
rate_improvement_hz = abs(stim_rate - objective['rate_hz']) - abs(tuned_rate - objective['rate_hz'])
kappa_improvement = abs(stim_kappa - objective['kappa']) - abs(tuned_kappa - objective['kappa'])

## Export: Metrics Fields

In [ ]:
metrics = {'artifact_id': 'etude_no_1', 'baseline_rate_hz': float(baseline_rate),
           'stimulus_rate_hz': float(stim_rate), 'tuned_rate_hz': float(tuned_rate),
           'baseline_kappa': float(baseline_kappa), 'stimulus_kappa': float(stim_kappa),
           'tuned_kappa': float(tuned_kappa), 'agsdr_gen': optimizer['gen'],
           'agsdr_pop': optimizer['pop']}

## Export: Metrics AGSDR Evidence

In [ ]:
metrics |= {'best_score': best_score, 'best_parameters': best_parameters,
            'tuning_status': tuning_status, 'same_model_unchanged': bool(same_model_unchanged),
            'rate_improvement_hz': float(rate_improvement_hz),
            'kappa_improvement': float(kappa_improvement)}

## Save JSON Artifacts

In [ ]:
jtfne.save_json(manifest, OUTPUT_DIR / 'manifest.json')
jtfne.save_json(validation, OUTPUT_DIR / 'validation_report.json')
jtfne.save_json(metrics, OUTPUT_DIR / 'metrics.json')

## Compute Artifact Hashes

In [ ]:
artifact_files = [OUTPUT_DIR / 'manifest.json', OUTPUT_DIR / 'validation_report.json',
                  OUTPUT_DIR / 'metrics.json', FIG_DIR / 'spectrolaminar.png']
hashes = {f.name: hashlib.sha256(f.read_bytes()).hexdigest() for f in artifact_files}
jtfne.save_json(hashes, OUTPUT_DIR / 'asset_hashes.json')

## Final Completion Message

In [ ]:
print(f'OK Etude No. 1 complete ({"SMOKE" if SMOKE else "FULL ETUDE"}) | '
      f'baseline {float(baseline_rate):.2f} -> tuned {float(tuned_rate):.2f} Hz '
      f'(target {objective["rate_hz"]}) | rate_improvement {float(rate_improvement_hz):.2f} Hz')

## Completion Scope

This Etude writes simulated/proxy artifacts only. It preserves `truth_safe_unverified`,
`computational_scaffold`, `laminar_proxy_no_pde`, and `physical_amplitude_claim_allowed=False`.

No physical mechanism claims, no field-solver validation, no solver-error estimates. The AGSDR
selected candidate is an optimizer choice over a proxy objective, not biological truth.